# Predicting Power Outage Durations: Does Weather Play a Role?

**Name(s)**: Nolan Lo, Mehul Kalsi, Jacob Sease

**Website Link**: [https://nolan-lo.github.io/Predicting_Power_Outage_Duration/](https://nolan-lo.github.io/Predicting_Power_Outage_Duration/)

In [16]:
import pandas as pd
import numpy as np
from pathlib import Path

import plotly.express as px
pd.options.plotting.backend = 'plotly'

# from dsc259r_utils import * # Feel free to uncomment and use this.

## Step 1: Introduction

The dataset we are working with records **major power outages** in the continental United States from January 2000 to July 2016. Each of the **1,534 rows** represents a single outage event, and the **56 columns** capture a wide range of information: when and where the outage occurred, its cause and duration, the number of customers affected, regional climate characteristics, electricity consumption patterns, and state-level economic and demographic indicators.

### Questions we brainstormed
1. **What factors most strongly predict how long a power outage will last?** (e.g., cause category, climate region, population density, time of year)
2. Which U.S. states or climate regions experience the most frequent and most severe outages?
3. Has the frequency or severity of outages changed over time (2000–2016)?

### Question we chose to investigate
> **What are the biggest factors that determine outage duration?**

We chose this question because outage duration directly impacts public safety and economic cost. By identifying the features most predictive of duration, utilities and policymakers could allocate resources more effectively. We will frame this as a **regression** problem in later steps, predicting `OUTAGE.DURATION` (in minutes) from features known at the time an outage begins.

## Step 2: Data Cleaning and Exploratory Data Analysis

In [17]:
# Load the data (skip metadata rows at top; row index 6 is the units row)
df = pd.read_excel('outage.xlsx', skiprows=[0, 1, 2, 3, 4, 6], header=0)
df = df.drop(columns=['variables'])  # drop the leftover label column

# --- Combine date + time columns into single Timestamp columns ---

# OUTAGE.START
df['OUTAGE.START'] = pd.to_datetime(
    df['OUTAGE.START.DATE']
) + pd.to_timedelta(
    df['OUTAGE.START.TIME'].astype(str), errors='coerce'
)

# OUTAGE.RESTORATION
df['OUTAGE.RESTORATION'] = pd.to_datetime(
    df['OUTAGE.RESTORATION.DATE']
) + pd.to_timedelta(
    df['OUTAGE.RESTORATION.TIME'].astype(str), errors='coerce'
)

# Drop the now-redundant individual date/time columns
df = df.drop(columns=[
    'OUTAGE.START.DATE', 'OUTAGE.START.TIME',
    'OUTAGE.RESTORATION.DATE', 'OUTAGE.RESTORATION.TIME',
])

print(f"Shape: {df.shape}")
df.head()

Shape: (1534, 54)


,OBS,YEAR,MONTH,U.S._STATE,POSTAL.CODE,NERC.REGION,CLIMATE.REGION,ANOMALY.LEVEL,CLIMATE.CATEGORY,CAUSE.CATEGORY,...,POPDEN_URBAN,POPDEN_UC,POPDEN_RURAL,AREAPCT_URBAN,AREAPCT_UC,PCT_LAND,PCT_WATER_TOT,PCT_WATER_INLAND,OUTAGE.START,OUTAGE.RESTORATION
0,1,2011,7.0,Minnesota,MN,MRO,East North Central,-0.3,normal,severe weather,...,2279.0,1700.5,18.2,2.14,0.6,91.592666,8.407334,5.478743,2011-07-01 17:00:00,2011-07-03 20:00:00
1,2,2014,5.0,Minnesota,MN,MRO,East North Central,-0.1,normal,intentional attack,...,2279.0,1700.5,18.2,2.14,0.6,91.592666,8.407334,5.478743,2014-05-11 18:38:00,2014-05-11 18:39:00
2,3,2010,10.0,Minnesota,MN,MRO,East North Central,-1.5,cold,severe weather,...,2279.0,1700.5,18.2,2.14,0.6,91.592666,8.407334,5.478743,2010-10-26 20:00:00,2010-10-28 22:00:00
3,4,2012,6.0,Minnesota,MN,MRO,East North Central,-0.1,normal,severe weather,...,2279.0,1700.5,18.2,2.14,0.6,91.592666,8.407334,5.478743,2012-06-19 04:30:00,2012-06-20 23:00:00
4,5,2015,7.0,Minnesota,MN,MRO,East North Central,1.2,warm,severe weather,...,2279.0,1700.5,18.2,2.14,0.6,91.592666,8.407334,5.478743,2015-07-18 02:00:00,2015-07-19 07:00:00


In [18]:

# --- Univariate: Distribution of outage duration (log scale) ---

dur = df['OUTAGE.DURATION'].dropna()

fig1 = px.histogram(
    dur,
    nbins=80,
    log_y=True,
    labels={'value': 'Outage Duration (minutes)', 'count': 'Count'},
    title='Distribution of Outage Duration (log-scaled y-axis)',
    color_discrete_sequence=['#636EFA'],
)

fig1.update_layout(
    showlegend=False,
    plot_bgcolor='white',
    yaxis=dict(gridcolor='lightgrey', title='Count (log scale)'),
    xaxis=dict(title='Outage Duration (minutes)'),
    bargap=0.05,
)

fig1.show()


**Univariate analysis – Outage Duration Distribution:** The histogram above shows that outage durations are extremely right-skewed: the vast majority of outages last under ~5,000 minutes (≈3.5 days), while a long tail extends past 50,000 minutes. This heavy skew suggests that a small number of catastrophic events (often severe weather) drive extraordinarily long outages, while most outages are resolved relatively quickly.

In [19]:

# --- Bivariate: Outage duration by cause category (box plot, log scale) ---

cause_duration_order = (
    df.groupby('CAUSE.CATEGORY')['OUTAGE.DURATION']
    .median()
    .sort_values(ascending=False)
    .index.tolist()
)

fig2 = px.box(
    df.dropna(subset=['OUTAGE.DURATION', 'CAUSE.CATEGORY']),
    x='CAUSE.CATEGORY',
    y='OUTAGE.DURATION',
    category_orders={'CAUSE.CATEGORY': cause_duration_order},
    color='CAUSE.CATEGORY',
    log_y=True,
    points='outliers',
    labels={
        'CAUSE.CATEGORY': 'Cause Category',
        'OUTAGE.DURATION': 'Outage Duration (minutes, log scale)',
    },
    title='Outage Duration by Cause Category',
    color_discrete_sequence=px.colors.qualitative.Safe,
)

fig2.update_layout(
    showlegend=False,
    xaxis_tickangle=-25,
    plot_bgcolor='white',
    yaxis=dict(gridcolor='lightgrey'),
)

fig2.show()


**Bivariate analysis – Duration vs. Cause Category:** The box plot reveals that **fuel supply emergencies** have by far the highest median outage duration, followed by severe weather events. In contrast, **intentional attacks** (e.g., vandalism/sabotage) tend to have very short durations. This supports our hypothesis that certain cause categories are strong predictors of outage length.

In [20]:

# --- Interesting Aggregate: Pivot table of mean outage duration by cause category and climate region ---

pivot = (
    df.dropna(subset=['OUTAGE.DURATION', 'CAUSE.CATEGORY', 'CLIMATE.REGION'])
    .pivot_table(
        values='OUTAGE.DURATION',
        index='CAUSE.CATEGORY',
        columns='CLIMATE.REGION',
        aggfunc='mean',
    )
    .round(0)
)

# Sort index by overall mean duration (descending)
pivot = pivot.loc[pivot.mean(axis=1).sort_values(ascending=False).index]

pivot


CLIMATE.REGION,Central,East North Central,Northeast,Northwest,South,Southeast,Southwest,West,West North Central
CAUSE.CATEGORY,,,,,,,,,
fuel supply emergency,10035.0,33971.0,14630.0,1.0,17482.0,NaN,76.0,6155.0,NaN
severe weather,3250.0,4435.0,4430.0,4838.0,4391.0,2663.0,11573.0,2928.0,2442.0
equipment failure,322.0,26435.0,216.0,702.0,296.0,554.0,114.0,525.0,61.0
public appeal,1410.0,733.0,2655.0,898.0,1164.0,2865.0,2275.0,2028.0,440.0
system operability disruption,2695.0,2610.0,774.0,141.0,866.0,169.0,329.0,364.0,NaN
intentional attack,346.0,2376.0,196.0,374.0,326.0,505.0,266.0,858.0,24.0
islanding,125.0,1.0,881.0,73.0,494.0,NaN,2.0,215.0,68.0


**Interesting Aggregate – Mean Duration by Cause × Climate Region:** The pivot table above shows mean outage duration (in minutes) for each combination of cause category and climate region. Fuel supply emergencies stand out with extremely high average durations in certain regions (e.g., East North Central), while intentional attacks consistently have short durations across all regions. The many `NaN` cells indicate cause–region combinations with no recorded outages, reflecting geographic concentration of certain outage types. This table highlights that **both cause category and climate region jointly influence outage length**, motivating their inclusion as features in our prediction model.

## Step 3: Assessment of Missingness

### NMAR Analysis

We believe the column **`DEMAND.LOSS.MW`** (missing in ~46% of rows) is likely **NMAR**. Our reasoning is that demand loss is only carefully measured and reported when the outage is significant enough to warrant detailed tracking. When the actual demand loss is very small or negligible (e.g., a brief localized outage), utilities may not bother recording the precise megawatt loss, so the value is left blank. In other words, the *probability that `DEMAND.LOSS.MW` is missing depends on its own (unobserved) value* — small demand losses are less likely to be recorded — which is the defining characteristic of NMAR.

**Additional data that could make it MAR:** If we obtained each utility's *reporting threshold* (the minimum MW loss that triggers mandatory reporting) and their internal *data completeness policies*, then the missingness could be explained by these external factors rather than the value itself, converting the missingness mechanism from NMAR to MAR.

### Missingness Dependency

We analyze the missingness of **`CUSTOMERS.AFFECTED`** (missing in ~29% of rows). We perform two permutation tests at the **α = 0.05 significance level**:

1. **Does the missingness of `CUSTOMERS.AFFECTED` depend on `CAUSE.CATEGORY`?** We expect yes — certain cause types like fuel supply emergencies are far less likely to have customer counts recorded. *Test statistic: Total Variation Distance (TVD).*
2. **Does the missingness of `CUSTOMERS.AFFECTED` depend on `TOTAL.PRICE`?** We expect no — the average retail electricity price in the state shouldn't directly influence whether customer counts are recorded. *Test statistic: absolute difference in group means.*

In [21]:

# ── Permutation Test 1: CUSTOMERS.AFFECTED missingness vs CAUSE.CATEGORY ──
# Test statistic: Total Variation Distance (TVD) of the distribution of
# CAUSE.CATEGORY when CUSTOMERS.AFFECTED is missing vs not missing.

n_repetitions = 1000

sub1 = df[['CUSTOMERS.AFFECTED', 'CAUSE.CATEGORY']].copy()
sub1['is_missing'] = sub1['CUSTOMERS.AFFECTED'].isnull()

# Observed distributions
dist_missing = sub1[sub1['is_missing']]['CAUSE.CATEGORY'].value_counts(normalize=True)
dist_present = sub1[~sub1['is_missing']]['CAUSE.CATEGORY'].value_counts(normalize=True)
# Align indices
all_cats = dist_missing.index.union(dist_present.index)
dist_missing = dist_missing.reindex(all_cats, fill_value=0)
dist_present = dist_present.reindex(all_cats, fill_value=0)

observed_tvd = (dist_missing - dist_present).abs().sum() / 2
print(f"Observed TVD: {observed_tvd:.4f}")

# Permutation test
tvds = []
labels = sub1['is_missing'].values
cause_vals = sub1['CAUSE.CATEGORY'].values

for _ in range(n_repetitions):
    shuffled = np.random.permutation(labels)
    d_miss = pd.Series(cause_vals[shuffled]).value_counts(normalize=True)
    d_pres = pd.Series(cause_vals[~shuffled]).value_counts(normalize=True)
    d_miss = d_miss.reindex(all_cats, fill_value=0)
    d_pres = d_pres.reindex(all_cats, fill_value=0)
    tvds.append((d_miss - d_pres).abs().sum() / 2)

tvds = np.array(tvds)
p_value_1 = (tvds >= observed_tvd).mean()
print(f"P-value (TVD permutation test): {p_value_1:.4f}")

# Plot empirical distribution of TVD
fig3 = px.histogram(
    x=tvds,
    nbins=50,
    labels={'x': 'TVD (permuted)', 'y': 'Count'},
    title='Permutation Test: Missingness of CUSTOMERS.AFFECTED vs CAUSE.CATEGORY',
    color_discrete_sequence=['#636EFA'],
)
fig3.add_vline(x=observed_tvd, line_dash='dash', line_color='red',
               annotation_text=f'Observed TVD = {observed_tvd:.3f}',
               annotation_position='top left')
fig3.update_layout(
    plot_bgcolor='white',
    yaxis=dict(gridcolor='lightgrey', title='Count'),
    xaxis=dict(title='Total Variation Distance'),
    showlegend=False,
)
fig3.show()


Observed TVD: 0.5574
P-value (TVD permutation test): 0.0000


In [22]:

# ── Permutation Test 2: CUSTOMERS.AFFECTED missingness vs TOTAL.PRICE ──
# Test statistic: absolute difference in mean TOTAL.PRICE when
# CUSTOMERS.AFFECTED is missing vs not missing.

sub2 = df[['CUSTOMERS.AFFECTED', 'TOTAL.PRICE']].dropna(subset=['TOTAL.PRICE']).copy()
sub2['is_missing'] = sub2['CUSTOMERS.AFFECTED'].isnull()

observed_diff = (
    sub2[sub2['is_missing']]['TOTAL.PRICE'].mean()
    - sub2[~sub2['is_missing']]['TOTAL.PRICE'].mean()
)
observed_diff_abs = abs(observed_diff)
print(f"Observed absolute difference in means: {observed_diff_abs:.4f}")

# Permutation test
diffs = []
price_vals = sub2['TOTAL.PRICE'].values
labels2 = sub2['is_missing'].values

for _ in range(n_repetitions):
    shuffled = np.random.permutation(labels2)
    diff = price_vals[shuffled].mean() - price_vals[~shuffled].mean()
    diffs.append(abs(diff))

diffs = np.array(diffs)
p_value_2 = (diffs >= observed_diff_abs).mean()
print(f"P-value (difference-in-means permutation test): {p_value_2:.4f}")

# Plot empirical distribution
fig4 = px.histogram(
    x=diffs,
    nbins=50,
    labels={'x': 'Abs Diff in Mean TOTAL.PRICE (permuted)', 'y': 'Count'},
    title='Permutation Test: Missingness of CUSTOMERS.AFFECTED vs TOTAL.PRICE',
    color_discrete_sequence=['#636EFA'],
)
fig4.add_vline(x=observed_diff_abs, line_dash='dash', line_color='red',
               annotation_text=f'Observed |diff| = {observed_diff_abs:.3f}',
               annotation_position='top right')
fig4.update_layout(
    plot_bgcolor='white',
    yaxis=dict(gridcolor='lightgrey', title='Count'),
    xaxis=dict(title='Absolute Difference in Mean TOTAL.PRICE (cents/kWh)'),
    showlegend=False,
)
fig4.show()


Observed absolute difference in means: 0.1177
P-value (difference-in-means permutation test): 0.5180


### Missingness Results

**Test 1 — `CUSTOMERS.AFFECTED` missingness vs `CAUSE.CATEGORY`:**
- Observed TVD ≈ 0.557, p-value ≈ 0.0 (out of 1,000 permutations, none produced a TVD as extreme as observed).
- **We reject the null hypothesis** that the missingness of `CUSTOMERS.AFFECTED` is independent of `CAUSE.CATEGORY`. The missingness **does depend on** the cause of the outage — e.g., fuel supply emergencies and intentional attacks are far more likely to have missing customer counts than severe weather events. This means `CUSTOMERS.AFFECTED` is **MAR** (Missing At Random) with respect to `CAUSE.CATEGORY`.

**Test 2 — `CUSTOMERS.AFFECTED` missingness vs `TOTAL.PRICE`:**
- Observed |diff in means| ≈ 0.118 cents/kWh, p-value ≈ 0.476.
- **We fail to reject the null hypothesis** that the missingness of `CUSTOMERS.AFFECTED` is independent of `TOTAL.PRICE`. The average electricity price in a state **does not appear to influence** whether customer counts are recorded.

## Step 4: Hypothesis Testing

**Null (H₀)**: The mean `OUTAGE.DURATION` is the same for severe-weather outages and non-severe-weather outages. Any observed difference is due to random chance.

**Alternative (H₁)**: Severe-weather outages have a **longer** mean duration than non-severe-weather outages (one-sided).

**Test statistic**: Difference in group means (mean duration of severe weather − mean duration of non-severe-weather).

**Significance level**: α = 0.05.

**Why these choices are appropriate**: Our central question asks what drives outage duration, and severe weather is the most common cause category. A one-sided test is appropriate because we have a directional hypothesis motivated by the EDA (the box plot showed severe weather has higher median duration than most other categories). The difference in means is a natural and interpretable test statistic for comparing the central tendency of two groups.

In [23]:

# ── Permutation Test: Severe weather vs non-severe-weather outage durations ──

hyp_df = df.dropna(subset=['OUTAGE.DURATION']).copy()
hyp_df['is_severe_weather'] = hyp_df['CAUSE.CATEGORY'] == 'severe weather'

# Observed test statistic
mean_sw = hyp_df[hyp_df['is_severe_weather']]['OUTAGE.DURATION'].mean()
mean_other = hyp_df[~hyp_df['is_severe_weather']]['OUTAGE.DURATION'].mean()
observed_stat = mean_sw - mean_other
print(f"Mean duration (severe weather):     {mean_sw:.1f} min")
print(f"Mean duration (non-severe-weather): {mean_other:.1f} min")
print(f"Observed difference in means:       {observed_stat:.1f} min")

# Permutation test (one-sided: severe weather > other)
n_perm = 10_000
durations = hyp_df['OUTAGE.DURATION'].values
labels_sw = hyp_df['is_severe_weather'].values
n_sw = labels_sw.sum()

perm_stats = np.empty(n_perm)
for i in range(n_perm):
    shuffled = np.random.permutation(labels_sw)
    perm_stats[i] = durations[shuffled].mean() - durations[~shuffled].mean()

p_value_hyp = (perm_stats >= observed_stat).mean()
print(f"\nP-value (one-sided): {p_value_hyp:.4f}")

# Visualization: empirical distribution of permuted test statistics
fig5 = px.histogram(
    x=perm_stats,
    nbins=80,
    labels={'x': 'Difference in Mean Duration (permuted)', 'y': 'Count'},
    title='Permutation Test: Severe Weather vs Non-Severe-Weather Outage Duration',
    color_discrete_sequence=['#636EFA'],
)
fig5.add_vline(
    x=observed_stat, line_dash='dash', line_color='red',
    annotation_text=f'Observed diff = {observed_stat:.0f} min',
    annotation_position='top left',
)
fig5.update_layout(
    plot_bgcolor='white',
    yaxis=dict(gridcolor='lightgrey', title='Count'),
    xaxis=dict(title='Difference in Mean Duration (severe weather − other)'),
    showlegend=False,
)
fig5.show()


Mean duration (severe weather):     3884.0 min
Mean duration (non-severe-weather): 1346.2 min
Observed difference in means:       2537.8 min

P-value (one-sided): 0.0000


### Hypothesis Test Conclusion

With an observed difference of ~2,538 minutes and a **p-value ≈ 0.0** (out of 10,000 permutations, none produced a difference as large as observed), we **reject the null hypothesis** at the α = 0.05 significance level.

The data provide strong evidence that severe-weather outages tend to have longer durations than non-severe-weather outages. This result aligns with our EDA findings and supports the inclusion of `CAUSE.CATEGORY` as a key feature in our duration prediction model. However, we note that this is an observational finding — we cannot claim a causal relationship from this test alone.

## Step 5: Framing a Prediction Problem

**Prediction problem**: Predict `OUTAGE.DURATION` (outage duration in minutes).

**Type**: **Regression** — `OUTAGE.DURATION` is a continuous numeric variable ranging from 0 to ~108,000 minutes.

**Response variable**: `OUTAGE.DURATION`. We chose this because outage duration is the most direct measure of how "bad" an outage is from a planning perspective, and our central question asks what factors most influence it.

**Evaluation metric**: **Root Mean Squared Error (RMSE)**. We chose RMSE over alternatives because:
- Unlike $R^2$, RMSE is on the same scale as the response variable (minutes), making it directly interpretable.
- Unlike Mean Absolute Error (MAE), RMSE penalizes large prediction errors more heavily, which is appropriate here since severely under-predicting a long outage is costlier than slightly mis-predicting a short one.

**Features available at time of prediction**: At the moment an outage begins, we would plausibly know:
- **Cause category** (`CAUSE.CATEGORY`) — the type of event causing the outage.
- **Climate region** (`CLIMATE.REGION`) — the geographic/climate zone.
- **Month** (`MONTH`) — time of year.
- **Climate anomaly level** (`ANOMALY.LEVEL`) — El Niño/La Niña index.
- **State** (`U.S._STATE` / `POSTAL.CODE`) — where the outage is occurring.
- **Population & urbanization** (`POPULATION`, `POPPCT_URBAN`) — demographic context.

We would **not** know `CUSTOMERS.AFFECTED`, `DEMAND.LOSS.MW`, or `OUTAGE.RESTORATION` at prediction time, since those are only known after or during the outage.

In [24]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

# Prepare the modeling dataframe — drop rows with missing target
model_df = df.dropna(subset=['OUTAGE.DURATION']).copy()
print(f"Rows available for modeling: {len(model_df)}")
print(f"Target stats:\n{model_df['OUTAGE.DURATION'].describe().round(1)}")


Rows available for modeling: 1476
Target stats:
count      1476.0
mean       2625.4
std        5942.5
min           0.0
25%         102.2
50%         701.0
75%        2880.0
max      108653.0
Name: OUTAGE.DURATION, dtype: float64


## Step 6: Baseline Model

In [25]:

# ── Baseline Model: Linear Regression with CAUSE.CATEGORY + MONTH ──

# Features for the baseline
baseline_features = ['CAUSE.CATEGORY', 'MONTH']

# Drop rows missing any of the baseline features or the target
baseline_df = model_df.dropna(subset=baseline_features + ['OUTAGE.DURATION']).copy()

X = baseline_df[baseline_features]
y = baseline_df['OUTAGE.DURATION']

# Train/test split (80/20), stratify not available for regression but fix seed
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Build pipeline:
# - CAUSE.CATEGORY (nominal, 7 unique values) → OneHotEncoder
# - MONTH (quantitative, 1-12) → left as-is (StandardScaler for consistency)
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore', drop='first'), ['CAUSE.CATEGORY']),
        ('num', StandardScaler(), ['MONTH']),
    ]
)

baseline_pipe = Pipeline([
    ('preprocessor', preprocessor),
    ('model', LinearRegression()),
])

baseline_pipe.fit(X_train, y_train)

# Evaluate on both sets
y_train_pred = baseline_pipe.predict(X_train)
y_test_pred = baseline_pipe.predict(X_test)

train_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred))
test_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))
train_r2 = r2_score(y_train, y_train_pred)
test_r2 = r2_score(y_test, y_test_pred)

print("=== Baseline Model Performance ===")
print(f"Train RMSE: {train_rmse:.1f} min  |  Train R²: {train_r2:.4f}")
print(f"Test  RMSE: {test_rmse:.1f} min  |  Test  R²: {test_r2:.4f}")
print(f"\n(Naive baseline — predicting mean for all: RMSE = {y_test.std():.1f} min)")


=== Baseline Model Performance ===
Train RMSE: 4916.3 min  |  Train R²: 0.1588
Test  RMSE: 7183.9 min  |  Test  R²: 0.1590

(Naive baseline — predicting mean for all: RMSE = 7846.6 min)


### Baseline Model Description

**Model**: Ordinary Least Squares (Linear Regression) implemented in a single `sklearn` `Pipeline`.

**Features used (2 total)**:
| Feature | Type | Encoding |
|---|---|---|
| `CAUSE.CATEGORY` | **Nominal** (7 unique categories) | `OneHotEncoder` (drop first → 6 indicator columns) |
| `MONTH` | **Quantitative** (1–12) | `StandardScaler` (centered, unit variance) |

**Performance**:
- **Test RMSE ≈ 7,184 minutes** (~5.0 days)
- **Test R² ≈ 0.159**

**Assessment**: This baseline model is **not good**. It explains only ~16% of the variance in outage duration, and its RMSE of ~7,184 minutes means predictions are off by roughly 5 days on average. While it does beat the naive strategy of always predicting the mean (RMSE ≈ 7,847 min), the improvement is modest. The large errors are expected since we are using only two features and a simple linear model that cannot capture non-linear relationships or interactions. This motivates exploring additional features and more flexible models in Step 7.

## Step 7: Final Model

### Feature Engineering Rationale

We add the following new engineered features on top of the baseline's `CAUSE.CATEGORY` and `MONTH`:

| New Feature | Source Column(s) | Transformation | Rationale |
|---|---|---|---|
| `CLIMATE.REGION` | `CLIMATE.REGION` | `OneHotEncoder` | Different climate regions face different weather patterns and infrastructure; our pivot table in Step 2 showed that mean duration varies substantially across regions. |
| `ANOMALY.LEVEL` | `ANOMALY.LEVEL` | `QuantileTransformer` | The El Niño/La Niña index captures oceanic-atmospheric conditions that influence storm severity. A quantile transform maps this to a uniform distribution, helping tree-based models split more effectively on extreme anomaly values. |
| `POPULATION` | `POPULATION` | `QuantileTransformer` | States with larger populations may have more complex grids but also more repair crews. The raw population is very right-skewed; quantile-transforming it normalizes the distribution so the model can better distinguish between high- and low-population states. |
| `POPPCT_URBAN` | `POPPCT_URBAN` | passthrough (no transform) | Higher urbanization correlates with denser infrastructure and faster repair access, which could reduce duration. As a percentage (0–100), it's already on a reasonable scale. |

**Model choice**: We use a **Random Forest Regressor** because:
- It naturally captures **non-linear relationships** and **feature interactions** (e.g., severe weather × certain climate regions may compound duration).
- It is robust to outliers — important given the heavy right-skew of `OUTAGE.DURATION`.
- It does not require feature scaling for its core splits, though our transforms still help distributional issues.

### Hyperparameters to tune

We will tune via `GridSearchCV` (5-fold CV):

| Hyperparameter | Candidates | Why |
|---|---|---|
| `n_estimators` | 100, 200, 300 | More trees generally reduce variance; we want enough for stable predictions without excessive training time. |
| `max_depth` | 5, 10, 20, None | Controls model complexity. Too shallow → underfitting; too deep → overfitting to training noise. |
| `min_samples_split` | 2, 5, 10 | Higher values prevent the tree from splitting on very small groups, acting as regularization. |

In [ ]:

from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import QuantileTransformer, FunctionTransformer

# ── Final Model: Random Forest with engineered features ──

# Use the SAME rows as the baseline (same index) so we can compare fairly
final_features = ['CAUSE.CATEGORY', 'MONTH', 'CLIMATE.REGION',
                  'ANOMALY.LEVEL', 'POPULATION', 'POPPCT_URBAN']

# Build a dataframe aligned with baseline_df's index, filling in new columns
final_df = baseline_df[['CAUSE.CATEGORY', 'MONTH']].copy()
for col in ['CLIMATE.REGION', 'ANOMALY.LEVEL', 'POPULATION', 'POPPCT_URBAN']:
    final_df[col] = df.loc[final_df.index, col]

# Drop rows where any new feature is missing (keeps test/train comparable)
final_df = final_df.dropna(subset=final_features)
y_final = y.loc[final_df.index]

print(f"Rows after dropping NaN in new features: {len(final_df)} (baseline had {len(baseline_df)})")

# Use the same random_state=42 split on this (possibly slightly smaller) set
X_train_f, X_test_f, y_train_f, y_test_f = train_test_split(
    final_df, y_final, test_size=0.2, random_state=42
)

# Define the preprocessing pipeline
final_preprocessor = ColumnTransformer(
    transformers=[
        ('cause', OneHotEncoder(handle_unknown='ignore', drop='first'), ['CAUSE.CATEGORY']),
        ('climate', OneHotEncoder(handle_unknown='ignore', drop='first'), ['CLIMATE.REGION']),
        ('month', 'passthrough', ['MONTH']),
        ('anomaly_qt', QuantileTransformer(output_distribution='uniform', random_state=42), ['ANOMALY.LEVEL']),
        ('pop_qt', QuantileTransformer(output_distribution='uniform', random_state=42), ['POPULATION']),
        ('urban', 'passthrough', ['POPPCT_URBAN']),
    ]
)

# Grid search over Random Forest hyperparameters
rf_pipe = Pipeline([
    ('preprocessor', final_preprocessor),
    ('model', RandomForestRegressor(random_state=42, n_jobs=-1)),
])

param_grid = {
    'model__n_estimators': [100, 200, 300],
    'model__max_depth': [5, 10, 20, None],
    'model__min_samples_split': [2, 5, 10],
}

grid_search = GridSearchCV(
    rf_pipe,
    param_grid,
    cv=5,
    scoring='neg_root_mean_squared_error',
    n_jobs=-1,
    verbose=1,
)

grid_search.fit(X_train_f, y_train_f)

print(f"\nBest hyperparameters: {grid_search.best_params_}")
print(f"Best CV RMSE: {-grid_search.best_score_:.1f} min")


Rows after dropping NaN in new features: 1471 (baseline had 1476)
Fitting 5 folds for each of 36 candidates, totalling 180 fits


/home/nlo/school/259-final/.venv/lib/python3.12/site-packages/sklearn/preprocessing/_data.py:2885: UserWarning: n_quantiles (1000) is greater than the total number of samples (941). n_quantiles is set to n_samples.
  warnings.warn(
/home/nlo/school/259-final/.venv/lib/python3.12/site-packages/sklearn/preprocessing/_data.py:2885: UserWarning: n_quantiles (1000) is greater than the total number of samples (941). n_quantiles is set to n_samples.
  warnings.warn(
/home/nlo/school/259-final/.venv/lib/python3.12/site-packages/sklearn/preprocessing/_data.py:2885: UserWarning: n_quantiles (1000) is greater than the total number of samples (941). n_quantiles is set to n_samples.
  warnings.warn(
/home/nlo/school/259-final/.venv/lib/python3.12/site-packages/sklearn/preprocessing/_data.py:2885: UserWarning: n_quantiles (1000) is greater than the total number of samples (941). n_quantiles is set to n_samples.
  warnings.warn(
/home/nlo/school/259-final/.venv/lib/python3.12/site-packages/sklearn/pr


Best hyperparameters: {'model__max_depth': 10, 'model__min_samples_split': 10, 'model__n_estimators': 100}
Best CV RMSE: 5032.8 min


In [ ]:

# ── Retrain best model on full training set & evaluate on held-out test set ──

best_pipe = grid_search.best_estimator_

y_train_pred_f = best_pipe.predict(X_train_f)
y_test_pred_f = best_pipe.predict(X_test_f)

final_train_rmse = np.sqrt(mean_squared_error(y_train_f, y_train_pred_f))
final_test_rmse = np.sqrt(mean_squared_error(y_test_f, y_test_pred_f))
final_train_r2 = r2_score(y_train_f, y_train_pred_f)
final_test_r2 = r2_score(y_test_f, y_test_pred_f)

print("=== Final Model Performance ===")
print(f"Train RMSE: {final_train_rmse:.1f} min  |  Train R²: {final_train_r2:.4f}")
print(f"Test  RMSE: {final_test_rmse:.1f} min  |  Test  R²: {final_test_r2:.4f}")

print("\n=== Comparison to Baseline ===")
print(f"Baseline Test RMSE: {test_rmse:.1f} min  →  Final Test RMSE: {final_test_rmse:.1f} min")
print(f"Baseline Test R²:   {test_r2:.4f}       →  Final Test R²:   {final_test_r2:.4f}")
print(f"RMSE improvement:   {test_rmse - final_test_rmse:.1f} min ({(test_rmse - final_test_rmse)/test_rmse*100:.1f}%)")

# ── Visualization: Actual vs Predicted scatter plot ──
fig6 = px.scatter(
    x=y_test_f,
    y=y_test_pred_f,
    labels={'x': 'Actual Duration (min)', 'y': 'Predicted Duration (min)'},
    title='Final Model: Actual vs Predicted Outage Duration (Test Set)',
    opacity=0.5,
)
# Add y=x reference line
max_val = max(y_test_f.max(), y_test_pred_f.max())
fig6.add_shape(type='line', x0=0, y0=0, x1=max_val, y1=max_val,
               line=dict(dash='dash', color='red'))
fig6.update_layout(
    plot_bgcolor='white',
    yaxis=dict(gridcolor='lightgrey'),
    xaxis=dict(gridcolor='lightgrey'),
)
fig6.show()


=== Final Model Performance ===
Train RMSE: 3755.1 min  |  Train R²: 0.5115
Test  RMSE: 6887.9 min  |  Test  R²: 0.2248

=== Comparison to Baseline ===
Baseline Test RMSE: 7183.9 min  →  Final Test RMSE: 6887.9 min
Baseline Test R²:   0.1590       →  Final Test R²:   0.2248
RMSE improvement:   296.0 min (4.1%)


### Final Model Description & Results

**Model**: Random Forest Regressor in a single `sklearn` `Pipeline`, selected via `GridSearchCV` (5-fold CV, scoring = negative RMSE).

**Features (6 total, with 4 new engineered features)**:

| Feature | Type | Transform | New? |
|---|---|---|---|
| `CAUSE.CATEGORY` | Nominal (7 categories) | `OneHotEncoder` (drop first → 6 indicators) | Baseline |
| `MONTH` | Quantitative (1–12) | Passthrough | Baseline |
| `CLIMATE.REGION` | Nominal (9 categories) | `OneHotEncoder` (drop first → 8 indicators) | **New** |
| `ANOMALY.LEVEL` | Quantitative | `QuantileTransformer` (uniform) | **New** |
| `POPULATION` | Quantitative | `QuantileTransformer` (uniform) | **New** |
| `POPPCT_URBAN` | Quantitative (0–100%) | Passthrough | **New** |

**Best hyperparameters** (from GridSearchCV):
- `n_estimators = 100`
- `max_depth = 10`
- `min_samples_split = 10`

The moderate depth (10) and larger minimum split size (10) indicate that some regularization helps — unrestricted trees overfit to the training set's extreme outlier durations.

**Performance comparison**:

| Metric | Baseline (Linear Regression) | Final (Random Forest) |
|---|---|---|
| Test RMSE | ~7,184 min | **~6,888 min** |
| Test R² | 0.159 | **0.225** |

The final model reduces test RMSE by ~296 minutes (~4%) and increases R² from 0.16 to 0.23. While the absolute improvement is modest, the R² gain of ~41% relative shows the model captures meaningfully more variance. The scatter plot above shows that predictions cluster closer to the diagonal (perfect prediction) line, though the model still struggles with extreme-duration outliers — a fundamental challenge given the heavy-tailed distribution of outage durations.

**Why the new features help** (from a data-generating perspective):
- **Climate region** captures regional infrastructure differences (age of grid, terrain, crew availability) and regional weather patterns that influence repair time.
- **Anomaly level** reflects oceanic-atmospheric conditions (El Niño/La Niña) that drive more intense storms, potentially prolonging outages.
- **Population** proxies for grid complexity and available repair resources — large states may have more crews but also more infrastructure to manage.
- **Urban percentage** relates to how densely built the infrastructure is; urban areas typically have faster restoration due to proximity of repair crews and equipment.

## Step 8: Fairness Analysis

In [ ]:

# ── Step 8: Fairness Analysis ──
# Question: Does our model perform worse for outages in "rural" states
# than in "urban" states?
# We binarize POPPCT_URBAN at the median (~77%) into two groups.

from sklearn.metrics import mean_squared_error

# Get predictions on the test set (already computed)
test_predictions = best_pipe.predict(X_test_f)

# Binarize: "urban" = POPPCT_URBAN >= median, "rural" = below median
threshold = X_test_f['POPPCT_URBAN'].median()
group_labels = (X_test_f['POPPCT_URBAN'] >= threshold).map({True: 'urban', False: 'rural'})

# Compute RMSE for each group
def group_rmse(y_true, y_pred, groups, group_name):
    mask = groups == group_name
    return np.sqrt(mean_squared_error(y_true[mask], y_pred[mask]))

rmse_urban = group_rmse(y_test_f.values, test_predictions, group_labels.values, 'urban')
rmse_rural = group_rmse(y_test_f.values, test_predictions, group_labels.values, 'rural')

observed_diff = rmse_rural - rmse_urban
print(f"Urban RMSE:  {rmse_urban:.1f} min")
print(f"Rural RMSE:  {rmse_rural:.1f} min")
print(f"Observed difference (rural − urban): {observed_diff:.1f} min")
print(f"Threshold (median POPPCT_URBAN in test set): {threshold:.1f}%")

# ── Permutation Test ──
# H₀: The model is fair — RMSE for rural and urban groups are roughly the same;
#      any difference is due to random chance.
# H₁: The model is unfair — RMSE for rural states is higher than for urban states.
# Test statistic: RMSE(rural) − RMSE(urban)
# Significance level: α = 0.05

np.random.seed(42)
n_permutations = 1000
perm_diffs = np.zeros(n_permutations)

for i in range(n_permutations):
    shuffled = np.random.permutation(group_labels.values)
    rmse_u = group_rmse(y_test_f.values, test_predictions, shuffled, 'urban')
    rmse_r = group_rmse(y_test_f.values, test_predictions, shuffled, 'rural')
    perm_diffs[i] = rmse_r - rmse_u

p_value = np.mean(perm_diffs >= observed_diff)
print(f"\nPermutation test p-value: {p_value:.4f}")
if p_value < 0.05:
    print("→ Reject H₀: Evidence the model is unfair (higher RMSE for rural states).")
else:
    print("→ Fail to reject H₀: No significant evidence of unfairness.")

# ── Visualization ──
fig7 = px.histogram(
    x=perm_diffs,
    nbins=50,
    title='Fairness Permutation Test: RMSE(Rural) − RMSE(Urban)',
    labels={'x': 'RMSE Difference (rural − urban)', 'y': 'Count'},
    opacity=0.75,
)
fig7.add_vline(x=observed_diff, line_dash='dash', line_color='red',
               annotation_text=f'Observed = {observed_diff:.1f}',
               annotation_position='top right')
fig7.update_layout(
    plot_bgcolor='white',
    xaxis=dict(gridcolor='lightgrey'),
    yaxis=dict(gridcolor='lightgrey'),
    showlegend=False,
)
fig7.show()


Urban RMSE:  4548.0 min
Rural RMSE:  8624.6 min
Observed difference (rural − urban): 4076.6 min
Threshold (median POPPCT_URBAN in test set): 84.7%

Permutation test p-value: 0.3880
→ Fail to reject H₀: No significant evidence of unfairness.


### Fairness Analysis: Urban vs. Rural States

**Groups:**
- **Urban states**: States with `POPPCT_URBAN` ≥ the test-set median (84.7%)
- **Rural states**: States with `POPPCT_URBAN` < the test-set median

**Evaluation metric:** RMSE (Root Mean Squared Error) — appropriate for our regression model.

**Hypotheses:**
- **Null (H₀):** Our model is fair. Its RMSE for rural states and urban states are roughly the same, and any observed difference is due to random chance.
- **Alternative (H₁):** Our model is unfair. Its RMSE for rural states is higher than its RMSE for urban states.

**Test statistic:** RMSE(rural) − RMSE(urban)
**Significance level:** α = 0.05

**Procedure:** We shuffled the urban/rural group labels 1,000 times and recomputed the RMSE difference each time. The p-value is the fraction of permuted differences that are ≥ the observed difference.

**Results:**
- Urban RMSE: 4,548.0 min
- Rural RMSE: 8,624.6 min
- Observed difference: 4,076.6 min
- **p-value: 0.388**

**Conclusion:** At α = 0.05, we **fail to reject the null hypothesis** (p = 0.388). Although the rural group has a numerically higher RMSE, this difference is not statistically significant — it could plausibly arise from random variation in which outage events happen to fall into each group. We do not have sufficient evidence to conclude that our model is unfair across urban/rural states.

## Export Plots as interactive HTML files.

In [ ]:

# ── Export all plots and tables for the website ──
import os
os.makedirs('assets', exist_ok=True)

# Export plotly figures as standalone HTML
fig1.write_html('assets/fig1_duration_hist.html', include_plotlyjs='cdn')
fig2.write_html('assets/fig2_duration_by_cause.html', include_plotlyjs='cdn')
fig3.write_html('assets/fig3_missingness_cause.html', include_plotlyjs='cdn')
fig4.write_html('assets/fig4_missingness_price.html', include_plotlyjs='cdn')
fig5.write_html('assets/fig5_hypothesis_test.html', include_plotlyjs='cdn')
fig6.write_html('assets/fig6_actual_vs_predicted.html', include_plotlyjs='cdn')
fig7.write_html('assets/fig7_fairness_test.html', include_plotlyjs='cdn')

# Export DataFrame head as HTML table
df_head_html = df.head().to_html(classes='dataframe', border=0, index=False)
with open('assets/df_head.html', 'w') as f:
    f.write(df_head_html)

# Export pivot table as HTML
pivot_html = pivot.to_html(classes='dataframe', border=0, na_rep='—')
with open('assets/pivot_table.html', 'w') as f:
    f.write(pivot_html)

print("Exported all assets to assets/ folder:")
for fname in sorted(os.listdir('assets')):
    print(f"  {fname}")


Exported all assets to assets/ folder:
  df_head.html
  fig1_duration_hist.html
  fig2_duration_by_cause.html
  fig3_missingness_cause.html
  fig4_missingness_price.html
  fig5_hypothesis_test.html
  fig6_actual_vs_predicted.html
  fig7_fairness_test.html
  pivot_table.html
